In [ ]:
#!/usr/bin/env python3
"""Loads the MMseqs2 outputs of step 02: the deduplicated sequences of every label, the cluster metrics along the identity thresholds and the per-sequence cluster assignment at every threshold.

Paths and parameters come from the run_mmseqs2 section of configs/param_configs.yaml; labels are named and ordered as in the query table (exact_species).
"""
###################################### DEPENDENCIES_AND_CONSTANTS
from IPython.display import display
import pandas as pd
from pathlib import Path
from pyprojroot import find_root, has_file
import yaml
DEFAULT_QUERY_TSV = "configs/query_configs.tsv"
YAML_PATH = "configs/param_configs.yaml"
###################################### CONFIG
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
config = yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}
query_table = pd.read_csv(PROJECT_ROOT / config.get("dw_uniprot_seqs", {}).get("query_table", DEFAULT_QUERY_TSV), sep="\t", dtype=str)
species_names = dict(zip(query_table["label"], query_table["exact_species"].fillna(query_table["label"])))
mmseqs_params = config["run_mmseqs2"]
dedup_dir, clusters_dir = PROJECT_ROOT / mmseqs_params["dedup_dir"], PROJECT_ROOT / mmseqs_params["clusters_dir"]
print(f"[PYTHON-INFO] dedup identity {mmseqs_params['dedup_min_identity']}, thresholds {mmseqs_params['cluster_identity_thresholds']}")

In [ ]:
###################################### LOAD
dedup_ids = {lbl: [line[1:].split()[0] for line in open(dedup_dir / f"{lbl}_dedup.fasta") if line.startswith(">")] for lbl in species_names if (dedup_dir / f"{lbl}_dedup.fasta").exists()}
dedup_params = {lbl: (dedup_dir / f"{lbl}_dedup.params").read_text().strip() if (dedup_dir / f"{lbl}_dedup.params").exists() else "unknown (built before the params sidecar)" for lbl in dedup_ids}
cluster_metrics = pd.concat({lbl: pd.read_csv(clusters_dir / lbl / f"{lbl}_cluster_metrics.tsv", sep="\t") for lbl in species_names if (clusters_dir / lbl / f"{lbl}_cluster_metrics.tsv").exists()}, names=["label", None]).reset_index(level=0)
cluster_assignments = {lbl: pd.read_csv(clusters_dir / lbl / f"{lbl}_all_clusters_summary.tsv", sep="\t", index_col="sequence_id", dtype=str) for lbl in species_names if (clusters_dir / lbl / f"{lbl}_all_clusters_summary.tsv").exists()}
display(pd.DataFrame({"species": species_names, "deduplicated sequences": {l: len(i) for l, i in dedup_ids.items()}, "dedup parameters": dedup_params, "assignment columns": {l: a.shape[1] for l, a in cluster_assignments.items()}}))
display(cluster_metrics)